# SetWW — reproducible T4 ABBA

Pinned code: `fda492acf93f67815a89c46120b362ca9821f56c`. Source-only archive, verified by manifest; no credentials,
task rows, model weights or compiler caches. No GitHub token is needed.

Choose **GPU T4 x2**, enable **Internet**, then **Run All**. Only GPU 0 is used.
The notebook executes the unchanged official harness with two repeats and all ten checks.
For final submission use public **Save & Run All**, then copy that saved version URL.

Prior verified reference: 14345 → 4203 MiB, 79.87 → 80.12 s, 10/10 checks,
own share min 6.6616%, 183 integration tests. These are reference numbers;
the cells below measure this new session and do not assume a speedup or a score.


In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path
from rich.console import Console
console = Console()
CODE_COMMIT = "fda492acf93f67815a89c46120b362ca9821f56c"
EXPECTED_SOLUTION_SHA = "6f722767b6723facf60b9ac6f4e0139027a2c548a087de702b75525b9940a1a4"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
repo = Path("/kaggle/working/setww-repro")
repo.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))) as archive:
    for member in archive.infolist():
        target = os.path.realpath(repo / member.filename)
        assert os.path.commonpath([str(repo), target]) == str(repo)
    archive.extractall(repo)
manifest = json.loads((repo / "SOURCE_MANIFEST.json").read_text())
for relative, expected in manifest.items():
    assert hashlib.sha256((repo / relative).read_bytes()).hexdigest() == expected, relative
digest = hashlib.sha256()
for path in sorted((repo / "solution").glob("*.py")):
    digest.update(path.name.encode())
    digest.update(path.read_bytes())
assert digest.hexdigest() == EXPECTED_SOLUTION_SHA
console.print({"code_commit": CODE_COMMIT, "source_files_verified": len(manifest)})


In [ ]:
import torch, triton
console.print({"torch":torch.__version__, "triton":triton.__version__,
               "gpu":torch.cuda.get_device_name(0), "visible_gpus":torch.cuda.device_count()})
assert "T4" in torch.cuda.get_device_name(0) and torch.cuda.device_count() == 1
assert torch.__version__.startswith("2.11.0") and triton.__version__ == "3.6.0"
with (repo / "install.log").open("w") as log:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao","xformers",
                    "flash-attn","liger-kernel","unsloth","unsloth-zoo"],
                   stdout=log,stderr=subprocess.STDOUT)
    install = subprocess.run([sys.executable,"-m","pip","install","-q",
        "soup-cli[train]==0.77.0","transformers==5.16.1","peft==0.20.0",
        "trl==0.29.1","bitsandbytes==0.50.2","accelerate==1.14.0","pytest"],
        stdout=log,stderr=subprocess.STDOUT)
console.print((repo / "install.log").read_text()[-2500:],markup=False)
assert install.returncode == 0


In [ ]:
for command,name,environment in (
    ([sys.executable,"task/prepare_data.py"],"prepare.log",dict(os.environ)),
    ([sys.executable,"-m","pytest","solution","tests","-q","--tb=short"],
     "integration_tests.log",dict(os.environ,SOUP_APPLY_SOLUTION="1")),
):
    with (repo / name).open("w") as log:
        checked = subprocess.run(command,cwd=repo,env=environment,
                                 stdout=log,stderr=subprocess.STDOUT)
    console.print((repo / name).read_text()[-3500:],markup=False)
    assert checked.returncode == 0, name


In [ ]:
with (repo / "run_task.log").open("w") as log:
    measured = subprocess.run([sys.executable,"harness/run_task.py","--repeats","2"],
                              cwd=repo,stdout=log,stderr=subprocess.STDOUT)
console.print((repo / "run_task.log").read_text()[-3500:],markup=False)
assert measured.returncode == 0
with (repo / "check_rules.log").open("w") as log:
    admitted = subprocess.run([sys.executable,"harness/check_rules.py"],
                              cwd=repo,stdout=log,stderr=subprocess.STDOUT)
console.print((repo / "check_rules.log").read_text(),markup=False)
assert admitted.returncode == 0, "Keep failed evidence; this run is not admitted"
scored = subprocess.run([sys.executable,"harness/score.py","runs/result.json"],
                        cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
console.print(scored.stdout,markup=False)
assert scored.returncode == 0


In [ ]:
from rich.table import Table
result_path = repo / "runs/result.json"
result = json.loads(result_path.read_text())
checks = json.loads((repo / "runs/checks.json").read_text())
assert checks["solution_sha256"] == EXPECTED_SOLUTION_SHA
assert checks["result_sha256"] == hashlib.sha256(result_path.read_bytes()).hexdigest()
assert len(checks["checks"]) == 10 and all(row["ok"] for row in checks["checks"])
table = Table("Metric","Base","Solution")
for title,key in (("Full command, s","wall_s"),("GPU peak, MiB","gpu_peak_mib"),
                  ("Eval loss","eval_loss")):
    table.add_row(title,str(result["summary"]["base"][key]),str(result["summary"]["solution"][key]))
console.print(table)
console.print("ALL TEN CHECKS PASSED",style="bold green")
console.print(result["summary"]["ratios"])
provenance = {"code_commit":CODE_COMMIT,"solution_sha256":EXPECTED_SOLUTION_SHA,
              "result_sha256":checks["result_sha256"],"versions":result["versions"]}
(repo / "PROVENANCE.json").write_text(json.dumps(provenance,indent=2))
destination = Path("/kaggle/working/setww-repro-evidence.zip")
allowed = {"result.json","checks.json","metrics.json","task.yaml","train.log",
           "trainer_state.json","adapter_config.json","rewind.jsonl"}
with zipfile.ZipFile(destination,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((repo / "runs").rglob("*")):
        if path.is_file() and not any(part in ("triton_cache","inductor_cache","__pycache__")
                                      for part in path.parts):
            if path.name in allowed or path.name.startswith("eval_") and path.suffix in (".json",".log"):
                archive.write(path,path.relative_to(repo).as_posix())
    for name in ("PROVENANCE.json","SOURCE_MANIFEST.json","integration_tests.log",
                 "run_task.log","check_rules.log"):
        archive.write(repo / name,name)
console.print({"evidence":str(destination),"sha256":hashlib.sha256(destination.read_bytes()).hexdigest()})
